# LAB 9 — In God We Trust, Everyone Else Bring the Data
## Part 1: Dataset Discovery & Exploration

**Dataset:** Synthetic Marketing Channel Performance Dataset  
**Source:** Self-generated (mirrors Kaggle 'Digital Marketing Campaign Performance' structure)  
**URL:** `data/marketing_data.csv` (see `generate_dataset.py` for generation logic)  
**Why this dataset:**  
Kaggle's real-world marketing datasets often require API auth or are behind walls. This dataset is generated with realistic channel economics (based on actual industry benchmarks for CTR, CVR, CPA, AOV) across 7 channels over 90 days. It gives us: multiple groups to compare, continuous + binary metrics, realistic noise, seasonal patterns — everything the lab needs.

**TLDR (plain English):** We're pretending to be a marketing analyst with a \$500K monthly budget. We have 90 days of daily data for 7 ad channels. Our job: figure out which channels actually perform better, and whether those differences are real or just random noise.

## Step 1 — Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from scipy.stats import ttest_ind, fisher_exact
try:
    from scipy.stats import false_discovery_control
    HAS_FDR = True
except ImportError:
    HAS_FDR = False
    print("scipy < 1.11 — will use manual BH adjustment")

import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})

OUTPUT_DIR = 'outputs/'
DATA_PATH  = 'data/marketing_data.csv'

print(f"pandas {pd.__version__}  scipy {stats.__version__}  numpy {np.__version__}")

## Step 2 — Load & Explore

In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=['date'])

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nData types:")
print(df.dtypes)
print("\nFirst 5 rows:")
df.head()

In [ ]:
print("Missing values:")
print(df.isnull().sum())

print("\nChannels:", df['channel'].unique().tolist())
print("Date range:", df['date'].min().date(), "to", df['date'].max().date())
print("Days per channel:")
print(df.groupby('channel').size())

In [ ]:
# Clean: replace inf values that can appear in cpa/roas for zero-cost or zero-conversion rows
df.replace([np.inf, -np.inf], np.nan, inplace=True)

print("Summary statistics:")
df[['impressions','clicks','conversions','cost','revenue','ctr','conversion_rate','cpa','roas']].describe().round(2)

## Step 3 — Key Marketing Metrics by Channel

**TLDR:** We aggregate the 90 days into totals per channel, then compute the 'big 6' marketing KPIs. Think of CPA as 'how much did it cost to get one customer' (lower = better) and ROAS as 'for every dollar spent, how many did we get back' (higher = better).

In [ ]:
# Aggregate totals per channel
agg = df.groupby('channel').agg(
    impressions=('impressions', 'sum'),
    clicks=('clicks', 'sum'),
    conversions=('conversions', 'sum'),
    cost=('cost', 'sum'),
    revenue=('revenue', 'sum'),
).reset_index()

# Calculate KPIs from aggregated totals
agg['ctr']              = (agg['clicks']      / agg['impressions'])  * 100
agg['conversion_rate']  = (agg['conversions'] / agg['clicks'])       * 100
agg['cpa']              = agg['cost']    / agg['conversions']
agg['roas']             = agg['revenue'] / agg['cost']
agg['profit']           = agg['revenue'] - agg['cost']
agg['profit_margin']    = (agg['profit']  / agg['revenue']) * 100

# SEO has zero cost — ROAS is infinite, handle gracefully
agg.replace([np.inf, -np.inf], np.nan, inplace=True)

summary_display = agg[['channel','impressions','clicks','conversions','cost','revenue',
                        'ctr','conversion_rate','cpa','roas','profit','profit_margin']].copy()
summary_display = summary_display.sort_values('roas', ascending=False, na_position='first')

pd.set_option('display.float_format', '{:,.2f}'.format)
print("Channel Performance Summary:")
summary_display

In [ ]:
# Visualise key aggregate metrics — 6-panel bar chart
metrics_to_plot = [
    ('cpa',             'CPA (Cost per Acquisition) $', True,  'tomato'),
    ('roas',            'ROAS (Revenue / Cost)',         False, 'steelblue'),
    ('conversion_rate', 'Conversion Rate %',             False, 'seagreen'),
    ('conversions',     'Total Conversions',             False, 'mediumpurple'),
    ('cost',            'Total Cost $',                  True,  'darkorange'),
    ('profit',          'Profit $',                      False, 'teal'),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, (col, label, lower_better, color) in enumerate(metrics_to_plot):
    data = agg[['channel', col]].dropna().sort_values(col, ascending=lower_better)
    ax = axes[i]
    bars = ax.barh(data['channel'], data[col], color=color, alpha=0.82, edgecolor='white')
    ax.set_xlabel(label)
    ax.set_title(label, fontweight='bold')
    if col == 'profit':
        ax.axvline(0, color='black', linewidth=1.0, linestyle='--')
    for bar, val in zip(bars, data[col]):
        ax.text(bar.get_width() * 1.01, bar.get_y() + bar.get_height()/2,
                f'{val:,.1f}', va='center', fontsize=9)

plt.suptitle('Marketing Channel Performance Overview — 90 Days', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'group_metrics_overview.png', bbox_inches='tight')
plt.show()
print("Saved: group_metrics_overview.png")

In [ ]:
# Distribution plots — daily CPA and daily conversion rate per channel
# (shows variability, not just averages — important for stats testing)

daily = df.copy()
daily['cpa']             = daily.apply(lambda r: r['cost']/r['conversions'] if r['conversions'] > 0 else np.nan, axis=1)
daily['conversion_rate'] = daily['conversions'] / daily['clicks'] * 100
daily.replace([np.inf, -np.inf], np.nan, inplace=True)

channels = sorted(daily['channel'].unique())
palette   = sns.color_palette('tab10', n_colors=len(channels))
ch_colors = dict(zip(channels, palette))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# CPA distributions
ax = axes[0]
for ch in channels:
    vals = daily[daily['channel']==ch]['cpa'].dropna()
    if len(vals) > 5:
        vals.plot.hist(ax=ax, alpha=0.45, bins=20, color=ch_colors[ch], label=ch, edgecolor='white')
ax.set_title('Daily CPA Distribution by Channel', fontweight='bold')
ax.set_xlabel('CPA ($)')
ax.legend(fontsize=8)

# Box plots — conversion rate
ax = axes[1]
data_for_box = [daily[daily['channel']==ch]['conversion_rate'].dropna().values for ch in channels]
bp = ax.boxplot(data_for_box, labels=channels, patch_artist=True, medianprops={'color':'black','linewidth':2})
for patch, ch in zip(bp['boxes'], channels):
    patch.set_facecolor(ch_colors[ch])
    patch.set_alpha(0.75)
ax.set_title('Daily Conversion Rate Distribution by Channel', fontweight='bold')
ax.set_ylabel('Conversion Rate (%)')
ax.set_xticklabels(channels, rotation=30, ha='right')

plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'group_distributions.png', bbox_inches='tight')
plt.show()
print("Saved: group_distributions.png")

In [ ]:
# Key observations
best_roas  = agg.dropna(subset=['roas']).sort_values('roas', ascending=False).iloc[0]
best_cpa   = agg.dropna(subset=['cpa']).sort_values('cpa').iloc[0]
best_cvr   = agg.sort_values('conversion_rate', ascending=False).iloc[0]
worst_cpa  = agg.dropna(subset=['cpa']).sort_values('cpa', ascending=False).iloc[0]

print("=== KEY OBSERVATIONS ===")
print(f"Best ROAS:      {best_roas['channel']} ({best_roas['roas']:.1f}x)")
print(f"Lowest CPA:     {best_cpa['channel']} (${best_cpa['cpa']:.2f})")
print(f"Best Conv Rate: {best_cvr['channel']} ({best_cvr['conversion_rate']:.1f}%)")
print(f"Highest CPA:    {worst_cpa['channel']} (${worst_cpa['cpa']:.2f})")
print(f"\nSEO_Organic has zero cost — excluded from ROAS/CPA comparisons (would be infinite/zero).")
print(f"Email dominates on conversion rate — its list is pre-qualified, not cold traffic.")